# Bias in Bios Joint-Control Experiment

This experiment compares Base, Leak, Cost, and Joint policies under one fixed protocol. Every policy starts from an empty knowledge state, acquires exactly 15 queries, uses `lambda_q = 0`, and is selected using validation accuracy. The held-out test metrics are target accuracy, macro-F1, conditional-probe leakage, cumulative synthetic cost, and gender-associated query rate.

In [ ]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import t
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


def find_repo_root(start_path):
    for candidate in [Path(start_path).resolve(), *Path(start_path).resolve().parents]:
        if (candidate / "assets" / "concepts" / "bias_in_bios.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from claq.analysis import (
    fit_conditional_probe,
    fixed_horizon_rollout,
    summarize_fixed_horizon,
)
from claq.core import (
    build_uncertainty_cost,
    file_sha256,
    load_answer_cache,
    load_run_bundle,
    make_cached_answer_loader,
    save_answer_cache,
    save_bundle_checkpoint,
)
from claq.models import ConceptAnswererMLP
from claq.training import HistorySamplingConfig, build_claq_models, fit_claq, seed_everything

In [ ]:
EXPERIMENT = "bias_in_bios_joint"
PROTOCOL_VERSION = 1
SEEDS = (0, 1, 2, 3, 4)
SCREENING_SEED = 0
HORIZON = 15
NUM_EPOCHS = 50
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
ACTOR_EPS = 1.0
LAMBDA_S_STAR = 0.4
LAMBDA_Q = 0.0
LAMBDA_C_CANDIDATES = (0.01, 0.03, 0.1, 0.2, 0.3)
MAX_VALIDATION_ACCURACY_DROP = 0.01
CALIBRATION_SEED = 1729
CALIBRATION_SIZE = 1_000
SPLITS = ("train", "validation", "test")
TEXT_COLUMN = "hard_text"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_dir = repo_root / "artifacts" / "data" / "bias_in_bios_sample"
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
labels_wide_path = sample_dir / "labels_wide_test_train_validation.csv"
qa_checkpoint_path = repo_root / "artifacts" / "models" / "concept_qa_bias_in_bios_concept_answerer_mlp_openai_labels.pt"
runs_dir = repo_root / "artifacts" / "runs"
models_dir = repo_root / "artifacts" / "models"
cache_dir = repo_root / "artifacts" / "concept_answers" / "bias_in_bios"
runs_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

seed_everything(SCREENING_SEED)
print({"device": str(device), "horizon": HORIZON, "seeds": SEEDS, "lambda_q": LAMBDA_Q})

In [ ]:
concepts_df = pd.read_csv(concepts_path)
concept_names = concepts_df["concept"].tolist()
labels_wide = pd.read_csv(labels_wide_path)

split_frames = {}
for split in SPLITS:
    samples = pd.read_csv(sample_dir / f"{split}.csv").reset_index(names="sample_row")
    labels = labels_wide.loc[labels_wide["split"].eq(split)]
    merged = samples.merge(
        labels[["sample_row", "source_index", *concept_names]],
        on=["sample_row", "source_index"],
        how="inner",
        validate="one_to_one",
    )
    if len(merged) != len(samples):
        raise ValueError(f"Merged {len(merged)} rows for {split}, expected {len(samples)}")
    split_frames[split] = merged

profession_names = (
    split_frames["train"][["profession", "profession_name"]]
    .drop_duplicates()
    .sort_values("profession")["profession_name"]
    .tolist()
)
num_professions = len(profession_names)
y_by_split = {
    split: frame["profession"].to_numpy(dtype=np.int64)
    for split, frame in split_frames.items()
}
s_by_split = {
    split: frame["gender"].to_numpy(dtype=np.int64)
    for split, frame in split_frames.items()
}

calibration_rng = np.random.default_rng(CALIBRATION_SEED)
train_permutation = calibration_rng.permutation(len(split_frames["train"]))
calibration_indices = train_permutation[:CALIBRATION_SIZE]
policy_train_indices = train_permutation[CALIBRATION_SIZE:]

print({
    "splits": {split: len(frame) for split, frame in split_frames.items()},
    "queries": len(concept_names),
    "classes": num_professions,
    "policy_train": len(policy_train_indices),
    "calibration": len(calibration_indices),
})

In [ ]:
checkpoint = torch.load(qa_checkpoint_path, map_location=device, weights_only=False)
model_config = dict(checkpoint["model_config"])
model_config["hidden_dims"] = tuple(model_config["hidden_dims"])
concept_embeddings = checkpoint["concept_embeddings"].to(device).float()
decision_threshold = float(checkpoint.get("decision_threshold", 0.5))
concept_answerer = ConceptAnswererMLP(**model_config).to(device)
concept_answerer.load_state_dict(checkpoint["model_state_dict"])
concept_answerer.eval()
encoder = SentenceTransformer(checkpoint["encoder_name"], device=str(device))


def build_text_concept_inputs(text_batch):
    repeated_text = text_batch.repeat_interleave(len(concept_names), dim=0)
    repeated_concepts = concept_embeddings.repeat(text_batch.size(0), 1)
    return torch.cat((repeated_text, repeated_concepts), dim=1)


@torch.no_grad()
def predict_scores_for_texts(texts, batch_size=256):
    embeddings = encoder.encode(
        list(texts),
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    embedding_tensor = torch.tensor(embeddings, device=device)
    parts = []
    for start in tqdm(range(0, len(embedding_tensor), batch_size), desc="Concept-QA"):
        text_batch = embedding_tensor[start : start + batch_size]
        logits = concept_answerer(build_text_concept_inputs(text_batch)).view(
            len(text_batch), len(concept_names)
        )
        parts.append(torch.sigmoid(logits).cpu())
    return torch.cat(parts)


cache_metadata = {
    "experiment": "bias_in_bios",
    "concept_count": len(concept_names),
    "qa_checkpoint": qa_checkpoint_path.name,
    "qa_checkpoint_sha256": file_sha256(qa_checkpoint_path),
}
cache_paths = {split: cache_dir / f"{split}_hard_answers.pt" for split in SPLITS}
for split in SPLITS:
    if not cache_paths[split].exists():
        scores = predict_scores_for_texts(split_frames[split][TEXT_COLUMN].fillna(""))
        answers = torch.where(scores >= decision_threshold, 1, -1)
        save_answer_cache(
            cache_paths[split],
            answers=answers,
            labels=torch.tensor(y_by_split[split]),
            sensitive_targets=torch.tensor(s_by_split[split]),
            metadata=cache_metadata,
        )

answer_caches = {
    split: load_answer_cache(path, expected_metadata=cache_metadata)
    for split, path in cache_paths.items()
}
calibration_soft = predict_scores_for_texts(
    split_frames["train"].iloc[calibration_indices][TEXT_COLUMN].fillna("")
)
print({split: tuple(cache["answers"].shape) for split, cache in answer_caches.items()})

In [ ]:
policy_train_cache = {
    key: value[policy_train_indices]
    for key, value in answer_caches["train"].items()
    if key in {"answers", "labels", "sensitive_targets"}
}
train_loader = make_cached_answer_loader(
    policy_train_cache,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=device.type == "cuda",
)
validation_loader = make_cached_answer_loader(
    answer_caches["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=device.type == "cuda",
)
sensitive_indices = torch.tensor(
    concepts_df.index[concepts_df["kind"].ne("utility")].tolist(),
    dtype=torch.long,
    device=device,
)
sensitive_mask = torch.zeros(len(concept_names), device=device)
sensitive_mask[sensitive_indices] = 1.0
uncertainty_cost = build_uncertainty_cost(
    calibration_soft,
    learned_mask=torch.ones(len(concept_names)),
    device=device,
)
COST_VECTOR_SHA256 = hashlib.sha256(
    uncertainty_cost.detach().cpu().numpy().tobytes()
).hexdigest()
cost_table = pd.DataFrame({
    "query_index": np.arange(len(concept_names)),
    "concept": concept_names,
    "gender_associated": sensitive_mask.cpu().numpy().astype(bool),
    "uncertainty_cost": uncertainty_cost.cpu().numpy(),
})
cost_table.to_csv(runs_dir / f"{EXPERIMENT}_cost_vector.csv", index=False)
display(cost_table.sort_values("uncertainty_cost", ascending=False).head(10))

In [ ]:
def load_or_train_policy(run_name, lambda_s, lambda_c, seed):
    seed_everything(seed)
    stem = f"{EXPERIMENT}_{run_name}_seed_{seed}"
    checkpoint_path = models_dir / f"{stem}_best.pt"
    history_path = runs_dir / f"{stem}_history.csv"

    if checkpoint_path.exists() and history_path.exists():
        bundle = load_run_bundle(
            checkpoint_path,
            device=device,
            max_queries=len(concept_names),
            num_classes=num_professions,
            actor_eps=ACTOR_EPS,
        )
        meta = bundle["meta"]
        expected = {
            "protocol_version": PROTOCOL_VERSION,
            "run_name": run_name,
            "seed": seed,
            "horizon": HORIZON,
            "lambda_s": float(lambda_s),
            "lambda_c": float(lambda_c),
            "lambda_q": LAMBDA_Q,
            "actor_eps": ACTOR_EPS,
            "cost_vector_sha256": COST_VECTOR_SHA256,
            "sensitive_conditioning": "conditional_y",
        }
        if any(meta.get(key) != value for key, value in expected.items()):
            raise ValueError(f"Checkpoint protocol mismatch for {checkpoint_path.name}")
        bundle.update({
            "run_name": run_name,
            "seed": seed,
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            "history": pd.read_csv(history_path),
            "best_epoch": int(meta["best_epoch"]),
        })
        return bundle

    actor, classifier, s_head = build_claq_models(
        max_queries=len(concept_names),
        num_classes=num_professions,
        device=device,
        actor_eps=ACTOR_EPS,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=LEARNING_RATE,
    )
    history_rows, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=validation_loader,
        model_clip=None,
        dictionary=torch.empty(0),
        answering_model=None,
        sens_idx=sensitive_indices,
        history_config=HistorySamplingConfig(0, 0, non_sensitive_only=False),
        clip_device=device,
        train_device=device,
        threshold_for_binarization=decision_threshold,
        lambda_s=lambda_s,
        lambda_c=lambda_c,
        sensitive_tau=1.0,
        sensitive_topk=1,
        num_epochs=NUM_EPOCHS,
        sensitive_target_mode="hard",
        cost_vector=uncertainty_cost,
        training_rollout_steps=HORIZON,
        lambda_q=LAMBDA_Q,
        designated_query_mask=sensitive_mask,
    )
    history = pd.DataFrame(history_rows).assign(run_name=run_name, seed=seed)
    history.to_csv(history_path, index=False)
    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    s_head.load_state_dict(best["s_head_state_dict"])
    save_bundle_checkpoint(
        checkpoint_path,
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        metadata={
            "experiment": EXPERIMENT,
            "protocol_version": PROTOCOL_VERSION,
            "run_name": run_name,
            "seed": seed,
            "horizon": HORIZON,
            "lambda_s": float(lambda_s),
            "lambda_c": float(lambda_c),
            "lambda_q": LAMBDA_Q,
            "actor_eps": ACTOR_EPS,
            "cost_vector_sha256": COST_VECTOR_SHA256,
            "sensitive_conditioning": "conditional_y",
            "best_epoch": int(best["epoch"]),
            "best_validation_accuracy": float(best["test_acc"]),
        },
    )
    bundle = load_run_bundle(
        checkpoint_path,
        device=device,
        max_queries=len(concept_names),
        num_classes=num_professions,
        actor_eps=ACTOR_EPS,
    )
    bundle.update({
        "run_name": run_name,
        "seed": seed,
        "lambda_s": lambda_s,
        "lambda_c": lambda_c,
        "history": history,
        "best_epoch": int(best["epoch"]),
    })
    return bundle

In [ ]:
validation_answers = answer_caches["validation"]["answers"].float()
validation_labels = answer_caches["validation"]["labels"].long()
validation_sensitive = answer_caches["validation"]["sensitive_targets"].long()

base_screen = load_or_train_policy("base", lambda_s=0.0, lambda_c=0.0, seed=SCREENING_SEED)
screening_runs = {0.0: base_screen}
for lambda_c in LAMBDA_C_CANDIDATES:
    name = f"screen_c_{lambda_c:g}".replace(".", "p")
    screening_runs[lambda_c] = load_or_train_policy(
        name, lambda_s=0.0, lambda_c=lambda_c, seed=SCREENING_SEED
    )

screening_rows = []
for lambda_c, run in screening_runs.items():
    rollout = fixed_horizon_rollout(
        actor=run["actor"],
        classifier=run["classifier"],
        answers=validation_answers,
        labels=validation_labels,
        sensitive_targets=validation_sensitive,
        cost_vector=uncertainty_cost,
        sensitive_mask=sensitive_mask,
        horizon=HORIZON,
        device=device,
        batch_size=BATCH_SIZE,
    )
    metrics = summarize_fixed_horizon(rollout, horizon=HORIZON, include_macro_f1=True)
    screening_rows.append({"lambda_c": lambda_c, **metrics})

lambda_c_screen = pd.DataFrame(screening_rows).sort_values("lambda_c").reset_index(drop=True)
base_accuracy = float(lambda_c_screen.loc[lambda_c_screen["lambda_c"].eq(0), "accuracy"].iloc[0])
eligible = lambda_c_screen[
    lambda_c_screen["lambda_c"].gt(0)
    & lambda_c_screen["accuracy"].ge(base_accuracy - MAX_VALIDATION_ACCURACY_DROP)
]
if eligible.empty:
    selected = lambda_c_screen[lambda_c_screen["lambda_c"].gt(0)].sort_values(
        ["accuracy", "mean_cumulative_cost"], ascending=[False, True]
    ).iloc[0]
else:
    selected = eligible.sort_values(
        ["mean_cumulative_cost", "accuracy"], ascending=[True, False]
    ).iloc[0]
LAMBDA_C_STAR = float(selected["lambda_c"])
lambda_c_screen.to_csv(runs_dir / f"{EXPERIMENT}_lambda_c_screen.csv", index=False)
print(f"Selected lambda_c={LAMBDA_C_STAR:g}")
display(lambda_c_screen)

In [ ]:
configurations = {
    "base": {"lambda_s": 0.0, "lambda_c": 0.0},
    "leak": {"lambda_s": LAMBDA_S_STAR, "lambda_c": 0.0},
    "cost": {"lambda_s": 0.0, "lambda_c": LAMBDA_C_STAR},
    "joint": {"lambda_s": LAMBDA_S_STAR, "lambda_c": LAMBDA_C_STAR},
}

runs = {}
for seed in SEEDS:
    runs[seed] = {}
    for run_name, weights in configurations.items():
        if seed == SCREENING_SEED and run_name == "base":
            run = base_screen
        else:
            run = load_or_train_policy(run_name, seed=seed, **weights)
        runs[seed][run_name] = run

print({seed: list(seed_runs) for seed, seed_runs in runs.items()})

In [ ]:
probe_train = {
    "answers": policy_train_cache["answers"].float(),
    "labels": policy_train_cache["labels"].long(),
    "sensitive": policy_train_cache["sensitive_targets"].long(),
}
probe_validation = {
    "answers": validation_answers,
    "labels": validation_labels,
    "sensitive": validation_sensitive,
}
test_data = {
    "answers": answer_caches["test"]["answers"].float(),
    "labels": answer_caches["test"]["labels"].long(),
    "sensitive": answer_caches["test"]["sensitive_targets"].long(),
}

rows = []
for seed, seed_runs in runs.items():
    for run_name, run in seed_runs.items():
        rollout_arguments = {
            "actor": run["actor"],
            "classifier": run["classifier"],
            "cost_vector": uncertainty_cost,
            "sensitive_mask": sensitive_mask,
            "horizon": HORIZON,
            "device": device,
            "batch_size": BATCH_SIZE,
        }
        train_rollout = fixed_horizon_rollout(
            answers=probe_train["answers"],
            labels=probe_train["labels"],
            sensitive_targets=probe_train["sensitive"],
            **rollout_arguments,
        )
        validation_rollout = fixed_horizon_rollout(
            answers=probe_validation["answers"],
            labels=probe_validation["labels"],
            sensitive_targets=probe_validation["sensitive"],
            **rollout_arguments,
        )
        test_rollout = fixed_horizon_rollout(
            answers=test_data["answers"],
            labels=test_data["labels"],
            sensitive_targets=test_data["sensitive"],
            **rollout_arguments,
        )
        test_metrics = summarize_fixed_horizon(
            test_rollout, horizon=HORIZON, include_macro_f1=True
        )
        probe = fit_conditional_probe(
            train_states=train_rollout["knowledge_states"],
            train_labels=train_rollout["labels"],
            train_sensitive=train_rollout["sensitive_targets"],
            validation_states=validation_rollout["knowledge_states"],
            validation_labels=validation_rollout["labels"],
            validation_sensitive=validation_rollout["sensitive_targets"],
            test_states=test_rollout["knowledge_states"],
            test_labels=test_rollout["labels"],
            test_sensitive=test_rollout["sensitive_targets"],
            num_classes=num_professions,
            random_state=seed,
        )
        rows.append({
            "dataset": "Bias in Bios",
            "run_name": run_name,
            "seed": seed,
            "horizon": HORIZON,
            **configurations[run_name],
            **test_metrics,
            "probe_leakage_bits": probe["probe_leakage_bits"],
            "probe_test_cross_entropy_bits": probe["test_cross_entropy_bits"],
            "probe_conditional_entropy_bits": probe["conditional_entropy_bits"],
            "probe_selected_c": probe["selected_c"],
            "probe_test_accuracy": probe["test_probe_accuracy"],
        })

results_by_seed = pd.DataFrame(rows)
display(results_by_seed)

In [ ]:
metrics = [
    "accuracy",
    "macro_f1",
    "probe_leakage_bits",
    "mean_cumulative_cost",
    "sensitive_query_rate",
]
t_critical = float(t.ppf(0.975, len(SEEDS) - 1))
summary_rows = []
for run_name in configurations:
    group = results_by_seed[results_by_seed["run_name"].eq(run_name)]
    row = {"dataset": "Bias in Bios", "run_name": run_name, "seeds": group["seed"].nunique()}
    for metric in metrics:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_ci95"] = t_critical * group[metric].std(ddof=1) / np.sqrt(len(group))
    summary_rows.append(row)
results_summary = pd.DataFrame(summary_rows)

results_by_seed.to_csv(runs_dir / f"{EXPERIMENT}_results_by_seed.csv", index=False)
results_summary.to_csv(runs_dir / f"{EXPERIMENT}_results_summary.csv", index=False)
manifest = {
    "experiment": EXPERIMENT,
    "horizon": HORIZON,
    "lambda_q": LAMBDA_Q,
    "lambda_s_star": LAMBDA_S_STAR,
    "lambda_c_star": LAMBDA_C_STAR,
    "checkpoints": {
        f"{run_name}_seed_{seed}": str(run["ckpt_path"].relative_to(repo_root))
        for seed, seed_runs in runs.items()
        for run_name, run in seed_runs.items()
    },
}
with open(models_dir / f"{EXPERIMENT}_checkpoints.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

display(results_summary)